# Satellite Land-Use Classification with Vision Transformers
Teaching an AI to look at a satellite photo and say what's on the ground: forest, farmland, city, highway, and more.

**Dataset:** EuroSAT (Sentinel-2 satellite tiles, 10 land-use classes)
**Model:** Vision Transformer (`google/vit-base-patch16-224`)
**Tools:** Hugging Face Transformers, PyTorch, Gradio

> Before running: Runtime → Change runtime type → **T4 GPU**

## 1. Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate gradio scikit-learn

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
print("GPU available:", torch.cuda.is_available())

## 2. Load the Dataset
EuroSAT RGB from Hugging Face. It downloads directly, no upload needed.

In [ ]:
from datasets import load_dataset

ds = load_dataset("blanchon/EuroSAT_RGB")
print(ds)

In [ ]:
names = ds["train"].features["label"].names
print(len(names), "classes:", names)

## 3. Explore the Data
A quick look at what the model will see: small 64x64 satellite tiles, one label each.

In [ ]:
import random
idx = random.sample(range(len(ds["train"])), 8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, idx):
    ex = ds["train"][i]
    ax.imshow(ex["image"]); ax.set_title(names[ex["label"]]); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Prepare the Data
We use a smaller random subset so training takes only a few minutes. The image processor resizes each tile to 224x224 and normalizes it for ViT.

In [ ]:
from transformers import AutoImageProcessor

ckpt = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(ckpt)

def tf(batch):
    inp = processor([im.convert("RGB") for im in batch["image"]], return_tensors="pt")
    inp["labels"] = batch["label"]
    return inp

train_raw = ds["train"].shuffle(seed=0).select(range(2000))
val_raw   = ds["validation"].shuffle(seed=0).select(range(500))
test_raw  = ds["test"].shuffle(seed=0).select(range(1000))

train_ds = train_raw.with_transform(tf)
val_ds   = val_raw.with_transform(tf)
test_ds  = test_raw.with_transform(tf)
print(len(train_ds), len(val_ds), len(test_ds))

## 5. Load the Pretrained Vision Transformer
ViT chops an image into 16x16 patches and uses self-attention to learn how the patches relate. The model already knows how to "see" from ImageNet. We swap in a new 10-class output layer for land types.

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    ckpt,
    num_labels=len(names),
    id2label=dict(enumerate(names)),
    label2id={n: i for i, n in enumerate(names)},
    ignore_mismatched_sizes=True,
)

## 6. Fine-Tune the Model

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="vit-eurosat",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=lambda p: {"acc": (np.argmax(p.predictions, 1) == p.label_ids).mean()},
)
trainer.train()

## 7. Evaluate on the Test Set
The test set was never seen during training, so this is the honest score.

In [ ]:
from sklearn.metrics import classification_report

out = trainer.predict(test_ds)
preds = out.predictions.argmax(1)
labels = out.label_ids
print("Test accuracy:", (preds == labels).mean())
print(classification_report(labels, preds, target_names=names))

### Confusion Matrix
Shows which classes the model mixes up. Bright cells off the diagonal are mistakes.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(confusion_matrix(labels, preds), display_labels=names).plot(
    ax=ax, xticks_rotation=60, colorbar=False)
plt.tight_layout(); plt.show()

## 8. Error Analysis: Where Does the Model Go Wrong?

In [ ]:
wrong = np.where(preds != labels)[0][:8]
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(test_raw[int(i)]["image"])
    ax.set_title(f"True: {names[labels[i]]}\nPred: {names[preds[i]]}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 9. Save the Model

In [ ]:
trainer.save_model("vit-eurosat")
processor.save_pretrained("vit-eurosat")

# Optional: back up to Google Drive so you don't lose it if Colab resets
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r vit-eurosat /content/drive/MyDrive/

## 10. Live Demo
Upload a satellite tile and the model tells you what it thinks it is.

In [ ]:
import gradio as gr
from transformers import pipeline

clf = pipeline("image-classification", model="vit-eurosat",
               device=0 if torch.cuda.is_available() else -1)

def predict(img):
    return {r["label"]: r["score"] for r in clf(img)}

gr.Interface(fn=predict, inputs=gr.Image(type="pil"),
             outputs=gr.Label(num_top_classes=3),
             title="Satellite Land-Use Classifier").launch(share=True)

## 11. Summary
- Fine-tuned a pretrained Vision Transformer on EuroSAT satellite tiles
- Reached high accuracy on unseen test images with only a few minutes of training
- Errors tend to happen between visually similar classes (e.g. Highway vs. River, crop types)

**Possible next steps:** compare against a ResNet baseline, try Swin Transformer, or use all 13 Sentinel-2 spectral bands.